# PDF central predictions versus PYTHIA8 CP5

Compare **pp** member-0 central predictions (mirrored by default) from CT18 or nCTEQ15HQ with standalone CP5 at 8.16 TeV. Run sequentially from the repository root using `py-env/bin/python`. The selected family refers to the proton reference stored in the corresponding nPDF artifact; it does not select the pPb `npdf_central`.

The existing nPDF workflow documents user-confirmed identical energy and jet/dijet selections. We retain that contract: anti-kT R=0.4, leading/subleading pT ≥ 50/40 GeV, |Δφ| ≥ 2π/3, and per-jet |etaCM| ≤ 1.9. CP5 acceptance is checked against the indexed input histograms. Other acceptances or unavailable pTave intervals fail explicitly.

## Configuration
Select the family and any exact shared nPDF intervals; multiple intervals run sequentially. The optional stored (50, 70) interval is also supported. Defaults match notebook 07. No cross section is inferred from the arbitrary relative normalizations.

In [ ]:
from pathlib import Path
import os
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / 'CMakeLists.txt').is_file() and (p / 'hist_analysis').is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from hist_analysis.config.histograms import NPDF_PTAVE_BINS
from hist_analysis.python.plot_labels import CMS_SIMULATION_HEADER, PP_HEADER_RIGHT

pdf_family = "CT18"  # Options: "CT18", "nCTEQ15HQ"
PDF_CENTRAL_KEY = 'pdf_central_mirrored'  # 'pdf_central' or 'pdf_central_mirrored'
PTAVE_BINS = list(NPDF_PTAVE_BINS)
JET_SELECTION = 'jetId'  # CP5 catalog production; generator dijets use the documented cuts
ETA_CUT_VALUE = 1.9  # only the supplied theory acceptance is supported
ETA_NORMALIZATION = 'integral'  # integral or bin_width (unit-area density)
CP5_FILE_OVERRIDE = os.environ.get('CP5_GEN_FILE')
OUTPUT_BASE = Path(os.environ.get('PDF_VS_CP5_OUTPUT_DIR',
    PROJECT_ROOT / 'hist_analysis/output/pdf_vs_cp5'))
CMS_HEADER_LEFT = CMS_SIMULATION_HEADER
CMS_HEADER_RIGHT = PP_HEADER_RIGHT
SAVE_PNG = False  # 200 DPI MPLHEP companion PNGs
DRAW_GRID = True
# Set any range to None for automatic limits.
PLOT_Y_LIMITS = {'etaCM': None, 'RFB': (0.5, 1.5)}
CP5_OVER_PDF_Y_LIMITS = {'etaCM': (0.5, 1.5), 'RFB': (0.5, 1.5)}


## Imports and initialization
Use the shared ROOT canvas renderer and its Matplotlib/MPLHEP CMS export. Both use the existing green CT18/EPPS21, magenta nCTEQ15HQ, and filled red CP5 styles. Figure geometry, typography, and marker sizes remain configurable through the shared style options.

In [ ]:
from hist_analysis.python.notebook_setup import load_root
ROOT = load_root(batch=True)
ROOT.gStyle.SetOptStat(0)  # Keep clean-kernel ROOT statistics boxes off comparison legends.
ROOT.TH1.AddDirectory(False)
from hist_analysis.python.pdf_cp5_comparison import (
    PDF_MODELS, PDF_COLORS, PDF_INTERVALS, PDF_CENTRAL_KEYS, load_cp5_source, prepare_pdf_cp5,
)
from hist_analysis.python.histogram_production import resolve_catalog_file
from hist_analysis.python.data_mc_plotting import draw_comparison_figure, accepted_eta_range, selection_lines
from hist_analysis.python.output_naming import OutputSelection, OutputDirectory
from hist_analysis.python.root_style import configure_cms_header, configure_mplhep_export
from hist_analysis.python.mplhep_plotting import MplhepStyle, save_canvas as save_mplhep_canvas
from hist_analysis.python.output_naming import mplhep_output_path
from IPython.display import display
import json
import math

MPLHEP_STYLE = MplhepStyle()  # figure size, fonts, markers, caps, and annotation position
configure_cms_header(CMS_HEADER_LEFT, CMS_HEADER_RIGHT)
configure_mplhep_export(False)  # explicit shared export below accepts configurable geometry
if pdf_family not in PDF_MODELS:
    raise ValueError(f'Unknown PDF family: {pdf_family}')
if PDF_CENTRAL_KEY not in PDF_CENTRAL_KEYS:
    raise ValueError(f'PDF_CENTRAL_KEY must be one of {PDF_CENTRAL_KEYS}')
PDF_DISPLAY_LABEL = pdf_family + (' (mirrored)' if PDF_CENTRAL_KEY == 'pdf_central_mirrored' else ' (original)')
if not PTAVE_BINS or len(set(map(tuple, PTAVE_BINS))) != len(PTAVE_BINS):
    raise ValueError('Select at least one unique pTave interval')
if any(tuple(b) not in PDF_INTERVALS for b in PTAVE_BINS):
    raise ValueError(f'Available exact intervals: {PDF_INTERVALS}')
if JET_SELECTION != 'jetId' or ETA_CUT_VALUE != 1.9:
    raise ValueError('The supplied theory comparison supports jetId / etaCM1p9 only')
for limits in (PLOT_Y_LIMITS, CP5_OVER_PDF_Y_LIMITS):
    if set(limits) != {'etaCM', 'RFB'}:
        raise ValueError('Y-axis limits require etaCM and RFB keys')
    for value in limits.values():
        if value is not None and (len(value) != 2 or
                not all(math.isfinite(v) for v in value) or value[0] >= value[1]):
            raise ValueError(f'Invalid y-axis limits: {value}')
CURVE_STYLES = {pdf_family: (PDF_COLORS[pdf_family], 24), 'PYTHIA CP5': (ROOT.kRed + 1, 20)}
OUTPUT_SELECTIONS = OutputSelection(JET_SELECTION,
    {'CM': ETA_CUT_VALUE, 'CP5GenCM': ETA_CUT_VALUE}, PTAVE_BINS)
OUTPUT_DIR = OutputDirectory(OUTPUT_BASE, OUTPUT_SELECTIONS)


## ROOT loading and prediction extraction

`PDF_CENTRAL_KEY` selects `pdf_central` (original) or `pdf_central_mirrored` (default). CT18 loads the selected key from the EPPS21 × CT18 artifact; nCTEQ15HQ loads the selected key from its own artifact. Metadata and the stored histogram title verify pp, CM, the exact pTave range, member 0, and scale 1. The optional 50–70 directory retains the first 50–60 component title, consistent with the stored width-weighted-combination metadata. The pPb/pp ratio and stored total bands are different products and are not used here. The mirrored spectrum is the stored reflected-bin average, M=(I+P)/2, with its stored marginal statistical errors; no additional smoothing is performed. Error bars are the central spectrum's stored statistical errors.

CP5 resolves the existing standalone catalog, preferring reusable inputs with source fallback. The retained raw `hGenDijetPtEtaCM` is projected over exact half-open pTave intervals.

In [ ]:
CP5_FILE = (Path(CP5_FILE_OVERRIDE).expanduser() if CP5_FILE_OVERRIDE else
            resolve_catalog_file('standalone__cp5__pp', selection=JET_SELECTION))
CP5_SOURCE = load_cp5_source(CP5_FILE, eta_cut=ETA_CUT_VALUE)
print('Standalone CP5:', CP5_FILE)


Standalone CP5: /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/root_files/produced_histograms/2026-10-22/jetId/root/standalone_cp5_pp_jetId_jetSelectionNA_etaCM1p5_1p6_1p7_1p8_1p9_2p4_2p5_3_ptAveBinsd36a1e87b37a3345.root


## Binning, normalization, and forward/backward construction

CP5's 0.1-wide eta bins are summed onto the common 0.2 theory edges with independent variances added in quadrature. No interpolation or partial-bin scaling is applied. The helper checks that the common range loses no nonzero yield. Stored partially accepted edge bins remain intact; displayed limits include those bins, as in notebook 07.

Full shapes use the same explicit normalization for both samples: `integral` divides by the bin-content sum; `bin_width` additionally divides by width, so the area is one. The uniform original theory-bin width is a global factor that cancels. As in existing notebooks, normalization rescales diagonal errors with a fixed integral; it does not estimate a covariance matrix.

F/B is constructed **before normalization** from the aligned raw full distributions using the shared ROOT folding and independent division:
$R_{FB}=F/B$, $sigma_R^2=sigma_F^2/B^2+F^2sigma_B^2/B^4$.
Zero backward yields retain ROOT's zero content/error numerically and are omitted from display copies. The selected PDF values are used directly; with mirrored input, populated PDF F/B bins are unity by construction of that stored spectrum. No computed ratio is forced to unity and no binomial errors are permitted.

The additional comparison ratios are $S_{CP5}/S_{PDF}$ for the normalized
full shapes and $(F/B)_{CP5}/(F/B)_{PDF}$ for the raw-yield F/B results. Both use
ROOT independent-error division, with no binomial treatment. Undefined source
F/B bins and zero PDF denominators are masked on display copies. Here PDF
means the selected family's **selected pp central histogram**, consistent with the overlays.

Mirroring correlates reflected bins. The existing independent-error ROOT F/B
convention is retained for both choices, using stored marginal bin errors;
it does not propagate the mirrored covariance matrix. Thus mirrored PDF F/B
can have nonzero displayed errors even though its populated values equal one.


In [ ]:
RESULTS = {}
for interval in PTAVE_BINS:
    RESULTS[tuple(interval)] = prepare_pdf_cp5(pdf_family, tuple(interval), CP5_SOURCE,
        eta_cut=ETA_CUT_VALUE, normalization=ETA_NORMALIZATION, project_root=PROJECT_ROOT,
        pdf_key=PDF_CENTRAL_KEY)
print('Prepared', len(RESULTS), 'intervals')


Prepared 7 intervals


## Full etaCM and F/B plots
Each interval produces separate full-distribution and F/B overlays in ROOT and MPLHEP PDF formats, with optional PNGs. Filenames encode family, CP5, observable, exact cuts, interval, normalization, and a unique run identifier. Selection sidecars preserve the complete identity.

`PLOT_Y_LIMITS['RFB']` sets the F/B overlay range, default `(0.5, 1.5)`.
`CP5_OVER_PDF_Y_LIMITS` independently controls the two CP5/PDF figures.
Each observable has an additional separate CP5/PDF ratio figure with a unity
reference line. Use `None` for automatic limits. These are display settings;
values outside a configured window remain unchanged in the ROOT histograms.


In [ ]:
FIGURES = []
CANVASES = []
for interval, result in RESULTS.items():
    selection = OutputSelection(JET_SELECTION,
        {'CM': ETA_CUT_VALUE, 'CP5GenCM': ETA_CUT_VALUE}, [interval])
    annotations = selection_lines(PROJECT_ROOT, interval, eta_cut=ETA_CUT_VALUE,
                                 direction='pp', selection=JET_SELECTION)
    for observable, key in (('etaCM', 'shapes'), ('RFB', 'fb')):
        stem = (f'pdf_{pdf_family}_vs_cp5_{observable}_{PDF_CENTRAL_KEY}_{ETA_NORMALIZATION}'
                f'_ptave_{interval[0]}_{interval[1]}')
        output = OUTPUT_DIR.file(stem + '.pdf', selection=selection)
        histograms = result[key]
        x_range = accepted_eta_range(next(iter(histograms.values())), ETA_CUT_VALUE,
                                    observable='full' if key == 'shapes' else 'fb')
        canvas = draw_comparison_figure(histograms, name='c_'+stem,
            x_title='#eta_{CM}^{dijet}' if key == 'shapes' else '|#eta_{CM}^{dijet}|',
            y_title=('Normalized yield' if ETA_NORMALIZATION == 'integral' else
                     '1/N dN/d#eta_{CM}^{dijet}') if key == 'shapes' else 'Forward / Backward',
            x_range=x_range, y_range=PLOT_Y_LIMITS[observable], styles=CURVE_STYLES,
            annotations=annotations, labels=lambda label: label.replace(pdf_family, PDF_DISPLAY_LABEL),
            reference=(key == 'fb'), grid=DRAW_GRID,
            output=output, save_png=SAVE_PNG)
        save_mplhep_canvas(canvas, output=mplhep_output_path(output),
            cms_header_left=CMS_HEADER_LEFT, cms_header_right=CMS_HEADER_RIGHT,
            style=MPLHEP_STYLE, save_png=SAVE_PNG)
        CANVASES.append(canvas)
        FIGURES.append(str(output))
        display(canvas)
        ratio_label = f'CP5 / {pdf_family}'
        ratio_stem = stem + '_cp5_over_pdf'
        ratio_output = OUTPUT_DIR.file(ratio_stem + '.pdf', selection=selection)
        ratio_canvas = draw_comparison_figure(
            {ratio_label: result['comparison_ratios'][key]}, name='c_'+ratio_stem,
            x_title='#eta_{CM}^{dijet}' if key == 'shapes' else '|#eta_{CM}^{dijet}|',
            y_title=ratio_label if key == 'shapes' else f'(F/B) CP5 / (F/B) {pdf_family}',
            x_range=x_range, y_range=CP5_OVER_PDF_Y_LIMITS[observable],
            styles={ratio_label: CURVE_STYLES['PYTHIA CP5']},
            annotations=annotations, labels=lambda label: label.replace(pdf_family, PDF_DISPLAY_LABEL),
            reference=True, grid=DRAW_GRID,
            output=ratio_output, save_png=SAVE_PNG)
        save_mplhep_canvas(ratio_canvas, output=mplhep_output_path(ratio_output),
            cms_header_left=CMS_HEADER_LEFT, cms_header_right=CMS_HEADER_RIGHT,
            style=MPLHEP_STYLE, save_png=SAVE_PNG)
        CANVASES.append(ratio_canvas)
        FIGURES.append(str(ratio_output))
        display(ratio_canvas)


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_7167893527940376758
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/pdf_vs_cp5/2026-10-29/jetId/root/jetId_etaCM1p9_etaCP5GenCM1p9_ptAve50to60_pdf_CT18_vs_cp5_etaCM_pdf_central_mirrored_integral_ptave_50_60_run1793315091433721000_8dec41771a45.pdf has been created


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_9181822645293574254
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/pdf_vs_cp5/2026-10-29/jetId/root/jetId_etaCM1p9_etaCP5GenCM1p9_ptAve50to60_pdf_CT18_vs_cp5_etaCM_pdf_central_mirrored_integral_ptave_50_60_cp5_over_pdf_run1793315091433721000_8dec41771a45.pdf has been created


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_1309185920070235788
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/pdf_vs_cp5/2026-10-29/jetId/root/jetId_etaCM1p9_etaCP5GenCM1p9_ptAve50to60_pdf_CT18_vs_cp5_RFB_pdf_central_mirrored_integral_ptave_50_60_run1793315091433721000_8dec41771a45.pdf has been created


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_7744818053606907318
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/pdf_vs_cp5/2026-10-29/jetId/root/jetId_etaCM1p9_etaCP5GenCM1p9_ptAve50to60_pdf_CT18_vs_cp5_RFB_pdf_central_mirrored_integral_ptave_50_60_cp5_over_pdf_run1793315091433721000_8dec41771a45.pdf has been created


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_1224747906836640989
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/pdf_vs_cp5/2026-10-29/jetId/root/jetId_etaCM1p9_etaCP5GenCM1p9_ptAve60to70_pdf_CT18_vs_cp5_etaCM_pdf_central_mirrored_integral_ptave_60_70_run1793315091433721000_8dec41771a45.pdf has been created


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_6253369098025080315
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/pdf_vs_cp5/2026-10-29/jetId/root/jetId_etaCM1p9_etaCP5GenCM1p9_ptAve60to70_pdf_CT18_vs_cp5_etaCM_pdf_central_mirrored_integral_ptave_60_70_cp5_over_pdf_run1793315091433721000_8dec41771a45.pdf has been created


... (18 outputs left out)

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_8678507496108751329
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/pdf_vs_cp5/2026-10-29/jetId/root/jetId_etaCM1p9_etaCP5GenCM1p9_ptAve200to300_pdf_CT18_vs_cp5_etaCM_pdf_central_mirrored_integral_ptave_200_300_run1793315091433721000_8dec41771a45.pdf has been created


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_8423523347698395751
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/pdf_vs_cp5/2026-10-29/jetId/root/jetId_etaCM1p9_etaCP5GenCM1p9_ptAve200to300_pdf_CT18_vs_cp5_etaCM_pdf_central_mirrored_integral_ptave_200_300_cp5_over_pdf_run1793315091433721000_8dec41771a45.pdf has been created


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_1205542593186013311
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/pdf_vs_cp5/2026-10-29/jetId/root/jetId_etaCM1p9_etaCP5GenCM1p9_ptAve200to300_pdf_CT18_vs_cp5_RFB_pdf_central_mirrored_integral_ptave_200_300_run1793315091433721000_8dec41771a45.pdf has been created


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_7682482607688764353
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/pdf_vs_cp5/2026-10-29/jetId/root/jetId_etaCM1p9_etaCP5GenCM1p9_ptAve200to300_pdf_CT18_vs_cp5_RFB_pdf_central_mirrored_integral_ptave_200_300_cp5_over_pdf_run1793315091433721000_8dec41771a45.pdf has been created


## Validation summary
Raw and aligned integrals, normalized integrals, bin counts, and zero denominators are recorded per interval. Every populated F/B error is checked against the independent-error formula. The JSON summary also records exact source paths and all generated ROOT-canvas PDFs; sibling MPLHEP PDFs share their names.

In [ ]:
from hist_analysis.config.theory import theory_file
SUMMARY = dict(pdf_family=pdf_family, pdf_key=PDF_CENTRAL_KEY,
    plot_y_limits=PLOT_Y_LIMITS, cp5_over_pdf_y_limits=CP5_OVER_PDF_Y_LIMITS,
    theory_file=str(theory_file(PDF_MODELS[pdf_family], PROJECT_ROOT)),
    cp5_file=str(CP5_FILE), cp5_key='hGenDijetPtEtaCM',
    validation=[r['validation'] for r in RESULTS.values()], figures=FIGURES)
summary_path = OUTPUT_DIR.file(f'pdf_{pdf_family}_vs_cp5_{PDF_CENTRAL_KEY}_{ETA_NORMALIZATION}_validation.json',
                               selection=OUTPUT_SELECTIONS)
summary_path.write_text(json.dumps(SUMMARY, indent=2) + '\n')
print(json.dumps(SUMMARY, indent=2))
print('Validation summary:', summary_path)


{
  "pdf_family": "CT18",
  "pdf_key": "pdf_central_mirrored",
  "plot_y_limits": {
    "etaCM": null,
    "RFB": [
      0.5,
      1.5
    ]
  },
  "cp5_over_pdf_y_limits": {
    "etaCM": [
      0.5,
      1.5
    ],
    "RFB": [
      0.5,
      1.5
    ]
  },
  "theory_file": "/Users/gnigmat/work/cms/jetAnalysis/hist_analysis/root_files/epps21_pPb_pp_pdf_scale_uncertainties_eta1p9_symmetric_cl68_fitresidual.root",
  "cp5_file": "/Users/gnigmat/work/cms/jetAnalysis/hist_analysis/root_files/produced_histograms/2026-10-22/jetId/root/standalone_cp5_pp_jetId_jetSelectionNA_etaCM1p5_1p6_1p7_1p8_1p9_2p4_2p5_3_ptAveBinsd36a1e87b37a3345.root",
  "cp5_key": "hGenDijetPtEtaCM",
  "validation": [
    {
      "ptave": [
        50,
        60
      ],
      "normalization": "integral",
      "pdf_key": "pdf_central_mirrored",
      "samples": {
        "CT18": {
          "raw_integral": 1485027.9189442173,
          "aligned_integral": 1485027.9189442173,
          "normalized_integral": 0.99